# LangChain과 Chroma를 활용한 RAG 구성

1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수도 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
3. 임베딩 --> 벡터 DB에 저장
4. 질문이 있을 때, 벡터 DB에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

## 1. 문서 쪼개서 document_list 생성
- 매뉴얼 정보 추가하기
- 마크다운 파일 1개를 1개의 document 객체로 만들어 리스트 반환 (문서 쪼개는 것 대신)

In [1]:
import os
import json
import re
from langchain.schema import Document # langchain_core.documents 로 변경 가능

# 2. 교육자료 로드 함수 (사용자님이 작성하신 코드)
def load_documents_from_metadata_json(
    md_dir="md_pdf",
    metadata_filepath="all_metadata.json"
):
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        all_metadata = json.load(f)

    docs = []
    for metadata in all_metadata:
        md_file_name = metadata.get("source")
        if not md_file_name:
            continue

        file_path = os.path.join(md_dir, md_file_name)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            
            docs.append(
                Document(
                    page_content=content,
                    metadata=metadata
                )
            )
        except FileNotFoundError:
            print(f"⚠️  경고: {file_path} 파일을 찾을 수 없습니다.")
            continue
            
    print(f"✅ 교육자료 Document 개수: {len(docs)}")
    return docs

# 3. 교육자료만 불러오기
education_docs = load_documents_from_metadata_json(
    md_dir="md_pdf",
    metadata_filepath="all_metadata.json"
)

✅ 교육자료 Document 개수: 49


- hwp 불러오기

In [2]:
from langchain_core.documents import Document
import json

def load_docs_from_json(input_path="split_hwp.json"):
    """
    JSON 파일을 langchain_core.documents.Document 리스트로 복원.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    document_list = [
        Document(page_content=item["page_content"], metadata=item["metadata"])
        for item in data
    ]
    print(f"✅ {len(document_list)}개의 문서를 불러왔습니다.")
    return document_list

In [4]:
document_list = load_docs_from_json("split_hwp.json")

✅ 34개의 문서를 불러왔습니다.


## 2. 임베딩

In [2]:
# 3. 임베딩 해주기 --> 백터DB에 저장 (Chroma 쓸꺼고 / 인메모리라 간단)
from dotenv import load_dotenv # 임베딩에 openai key 인자가 있어서 환경변수를 로드해줘야 함
from langchain_openai import OpenAIEmbeddings 

# 환경변수 불러옴
load_dotenv()

# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large') # 임베딩 모델을 large로 바꿔주기

## 3. 벡터DB 생성 (Pinecone) 

In [3]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'finance-new-index' # finance-new-index : 정산지침+교육자료(매뉴얼ppt) / finance-index: 매뉴얼ppt+정산지침
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

/Users/dayeayim/.pyenv/versions/inflearn-streamlit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
'''
# 존재하는 인덱스를 완전 삭제
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"Deleted index: {index_name}")

Deleted index: finance-new-index


In [4]:
# 데이터를 추가(insert)할 때는 `from_documents()`
# 데이터를 추가한 이후에는 `from_existing_index()`를 사용 (데이터 추가없이 검색만)
#database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name) 
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

In [6]:
'''
# 교육자료 추가 업데이트
print("🚀 교육자료 업로드를 시작합니다... (잠시만 기다려주세요)")
database.add_documents(education_docs)
print("🎉 교육자료 업로드가 완료되었습니다!")
'''

🚀 교육자료 업로드를 시작합니다... (잠시만 기다려주세요)
🎉 교육자료 업로드가 완료되었습니다!


In [7]:
import os
from pinecone import Pinecone

index_name = "finance-new-index"
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 1️⃣ 인덱스 상태 확인
index_info = pc.describe_index(index_name)
print(f"📊 인덱스 상태: {index_info.status}")

# 2️⃣ 인덱스 통계(벡터 개수) 확인
index_stats = pc.Index(index_name).describe_index_stats()
print(f"📦 '{index_name}' 인덱스 요약:")
print(index_stats)

# 3️⃣ 깔끔하게 개수만 출력하고 싶다면
vector_count = index_stats["total_vector_count"]
print(f"✅ 현재 '{index_name}'에는 {vector_count:,}개의 벡터가 저장되어 있습니다.") # 83개

📊 인덱스 상태: {'ready': True, 'state': 'Ready'}
📦 'finance-new-index' 인덱스 요약:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 24 Dec 2025 03:01:26 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-id': '5187085361901270119',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 83}},
 'storageFullness': 0.0,
 'total_vector_count': 83,
 'vector_type': 'dense'}
✅ 현재 'finan

## 4. 문서 검색 및 답변
- 벡터DB에서 적합한 문서 찾고 LLM에 문서와 질의를 주면서 답변을 요청(RetrievalQA)
- RetrievalQA: 데이터를 검색(Retrieval)한 다음에 질문(Question)하고 답변(Answer) 할 것이다

In [8]:
# 문서를 가지고 왔으니까 질의를 해봐야 한다 
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(model = 'gpt-4o')
llm = ChatOpenAI(model = 'gpt-5.1')

In [9]:
# Retrieval된 데이터는 LangChain에서 제공하는 프롬프트("rlm/rag-prompt") 사용해서 답변해보기
from langchain import hub 
from langchain.prompts import ChatPromptTemplate

# 기본 RAG 프롬프트 가져오기
base_prompt = hub.pull("rlm/rag-prompt")

# 시스템 역할 지시 추가해서 새 템플릿 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 재무팀에서 근무하는 해외무역관 정산 전문가입니다. "
               "[context]를 참고해서 사용자의 질문에 답변해주세요. "
               "[context]에 없는 정보는 지어내지 말고, 자료에 없음이라고 답하세요."
               ),
    *base_prompt.messages  # 기존 RAG 메시지 구조 유지
])

#### retrieve된 문서의 순서 변경하기
1. origin_pdf == "해외조직망정산지침"(A) 인 문서를 항상 앞으로 모은다.
2. 나머지 문서들(B 포함)은 뒤에 둔다.
3. 같은 그룹(A끼리, 그 외끼리) 는 원래 retriever가 준 순서를 그대로 유지한다.
4. A가 하나도 없는 경우는 기존 순서를 그대로 사용한다.

In [10]:
from typing import List, Any
from langchain_core.retrievers import BaseRetriever  # 버전에 따라 langchain.retrievers 일 수도 있음
from langchain.schema import Document

PRIORITY_ORIGIN = "정산지침"

class PriorityRetriever(BaseRetriever):
    base_retriever: Any

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        docs = self.base_retriever.get_relevant_documents(query)
        docs.sort(
            key=lambda d: 0 if d.metadata.get("origin_pdf") == PRIORITY_ORIGIN else 1
        )
        return docs

    async def _aget_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        docs = await self.base_retriever.aget_relevant_documents(query)
        docs.sort(
            key=lambda d: 0 if d.metadata.get("origin_pdf") == PRIORITY_ORIGIN else 1
        )
        return docs

In [11]:
# 답변생성 (QA 체인 만들기)
#-- RetrievalQA를 통해 LLM에 전달
#-- RetrievalQA는 create_retrieval_chain으로 대체됨
#-- 실제 ChatBot 구현 시 create_retrieval_chain으로 변경하는 과정을 볼 수 있음

from langchain.chains import RetrievalQA 

base_retriever = database.as_retriever( # 벡터DB
    search_kwargs={"k": 3}) # 유사도로 몇 개 문서 찾을 것인지

priority_retriever = PriorityRetriever(base_retriever = base_retriever)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = priority_retriever, # 위에서 만든 retriever (DB에서 문서 검색)
    chain_type_kwargs = {"prompt": prompt}, # 위에서 만든 prompt (정산 전문가)
    return_source_documents=True # 문서 소스 포함
)

In [12]:
query = '상품권 구매 시 정산은 어떻게 해야해?'

In [13]:
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

/var/folders/ql/nhk1fr510sv10fw11qxx_y3c0000gn/T/ipykernel_2368/2694196770.py:11: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = self.base_retriever.get_relevant_documents(query)


In [14]:
answer

'자료에 상품권(기프트카드) 구매·정산 관련 규정이나 회계처리 예시는 제시되어 있지 않습니다.  \n따라서 상품권 구매 시 정산방법은 자료에 없음입니다.'

In [15]:
ai_message["source_documents"]

[Document(id='200a19f6-bb8c-4fa1-8992-5d6145539e42', metadata={'images': ['040_img0.png'], 'origin_pdf': '교육자료', 'page_num': 40, 'source': '040.md'}, page_content='11.1.1 현금 또는 예금장부에 입금하고 신용카드장부에서 지출하는 경우의 실제 사례\n===============================================\n\n  \n\n- 무역관 문의: 신용카드 결제내역 일부 중 개인 물품이 실수로 포함되어 결제가 되었습니다. 어떻게 정산  \n처리를 해야할까요?  \n- 무역관 상황: 결제 100불 중 회사분 80불, 개인분 20불 / 카드결제일 : 6월 11일 / 오지출 발견일 : 6월 12일 /  \n신용카드 대금 납부일 : 6월 25일  \n- 처리방법:\n\n  \n\n① (6.11.) 신용카드장부 – 20불 이체(출금) 처리,  \n80불은 기존 실행사업, 비목으로 집행  \n② (6.12.) 무역관 계좌에 20불 입금,  \n송금통화예금장부 - 이체(입금) 처리  \n③ (6.25.) 신용카드장부 – 20불 이체(입금) 처리,  \n송금통화예금장부 – 20불 이체(출금) 처리  \n④ (6.25.) 80불에 대한 카드 대금 처리 : 80불 대체  \n송금통화예금(출금) → 신용카드 장부(입금)\n\n• 현금 또는 예금장부에 입금하고 신용카드장부에서 지출하는 경우의 실제 사례\n=========================================='),
 Document(id='5d545bec-e075-4348-ab82-b88f9b150d5f', metadata={'origin_pdf': '정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)3.  정산 시기 가. 자금정산은 발생주의 원칙에 의거, 집행일에 정산하여

In [22]:
answer

'자료에 따르면, 상품권은 공용카드 적립포인트를 유가증권으로 교환해 받은 경우에 한해 사용·정산방법이 규정되어 있습니다.  \n1) 포인트로 교환한 상품권 가액 전액을 교환 시점에 현금장부상 ‘잡이익’으로 수입 처리하고,  \n2) 해당 상품권을 실제 집행할 때는 ‘현금 집행 시와 동일한 방식’으로 비목별로 정산(영수증·사용내역 첨부, 집행일 기준 발생주의 정산)하면 됩니다.  \n\n포인트가 아닌 일반 예산으로 상품권을 직접 구매·정산하는 방법은 자료에 없음입니다.'

In [23]:
ai_message["source_documents"]

[Document(id='5d545bec-e075-4348-ab82-b88f9b150d5f', metadata={'origin_pdf': '정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)3.  정산 시기 가. 자금정산은 발생주의 원칙에 의거, 집행일에 정산하여야 한다.  1) 차분기에 결제일이 도래할 분기말 카드사용분은 카드슬립과 영수증만으로 해당분기에 정산한다.   2) 카드대금 이외의 일반청구서도 회계연도 이월집행 및 정산이 불가하므로 연도말에 발급된 청구서는 수표발급과 함께 4/4분기에 정산하여야 하며, 4/4분기말 기준으로 동 자금이 인출되지 않았을 경우에는 자금잔액명세서 상 미결제수표 대금으로 차액을 설명하여야 한다.  3) 전화요금 등 공공요금의 경우도 12월에 발급된 청구서의 경우, 납기 마감일이 익년도일 경우라도 4/4분기에 해당요금을 지불하고 정산을 종료하여야 한다. 나. 계약조건에 의한 경우를 제외하고는 집행경비를 2개 이상의 분기에 분할하여 정산할 수 없다. 다. 모든 자금은 지원받은 당해 회계연도 내에 집행하고 정산하여야 하며, 연말까지 집행, 정산하지 아니하는 경우에는 불용액으로 처리, 회수하여야 한다.4.  지적사항에 대한 조치 가. 정산사항에 대한 지적, 보완, 참고사항 등을 통보받은 경우에는 성실한 내용으로 보고하여야 하며, 보고내용은 통보 도착일로부터 20일 이내에 본사에 도착하여야 한다. 나. 금액수정을 통보받은 경우에는 정산결과 통보공문 접수일을 기준으로 차액을 예금 또는 현금으로 입금하고, 해당 장부에 입금내역을 기장하여야 한다.5. 관련법규 및 지침 준수 회계관계직원의 책임에 관한 법률, 부정청탁 및 금품수수 등 금지에 관한 법률, 임직원행동강령 등 관련 법, 규정, 지침 등을 준수하고 이를 위반하는 행위를 하지 않도록 한다.'),
 Document(id='ff37e503-6ed0-4323-8e47-9bb08eb85a7f', me

In [16]:
import re
from pathlib import Path

# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장 
# 객체에서는 속성에 점(.)을 사용하여 접근
# --- QA 실행 ---
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

# -----------------------------
# 1) page_num 포맷 함수
# -----------------------------
def _format_page_token(page_val):
    """
    포맷 규칙:
    - 숫자(int) 또는 '7', '7.0', '07.00' 등 숫자 형태 → '7p'
    - 문자 섞인 경우 원문 그대로 유지 ('2조 목적' 등)
    """
    if page_val is None:
        return None

    # int → '3p'
    if isinstance(page_val, int):
        return f"{page_val}p"

    s = str(page_val).strip()
    if not s:
        return None

    # float 형태 문자열 ('7.0', '3.00') → int로 변환 후 p
    try:
        num = float(s)
        if num.is_integer():
            return f"{int(num)}p"
    except ValueError:
        pass

    # 순수 숫자 문자열 ('12') → '12p'
    if re.fullmatch(r"\d+", s):
        return f"{s}p"

    # 문자 섞임 ('2조 목적', 'p.12' 등) → 그대로
    return s

# -----------------------------
# 2) 순서 보존 중복 제거
# -----------------------------
def _dedup_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# -----------------------------
# 3) 파일명 정제
# -----------------------------
def _resolve_pdf_name(metadata: dict):
    name = (metadata or {}).get("origin_pdf") or (metadata or {}).get("source") or "알 수 없음"
    try:
        return Path(str(name)).name   # 경로에서 파일명만 추출
    except Exception:
        return str(name)

# -----------------------------
# 4) source_documents에서 출처 문자열 만들기
# -----------------------------
docs = ai_message.get("source_documents", []) or []

origins_order = []       # 파일 등장 순서
pages_by_origin = {}     # {파일명: [페이지토큰...]}
seen_pairs = set()       # (파일명, 페이지토큰) 중복 방지

# k=3이니까 상위 3개만 쓰고 싶으면 [:3], 2개만이면 [:2]
for d in docs[:3]:
    md = getattr(d, "metadata", {}) or {}
    origin = _resolve_pdf_name(md)
    page_token = _format_page_token(md.get("page_num"))
    if not page_token:
        continue

    pair = (origin, page_token)
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)

    if origin not in pages_by_origin:
        pages_by_origin[origin] = []
        origins_order.append(origin)

    if page_token not in pages_by_origin[origin]:
        pages_by_origin[origin].append(page_token)

# 최종 출처 문자열 구성
if origins_order:
    parts = []
    for origin in origins_order:
        pages = _dedup_preserve_order(pages_by_origin.get(origin, []))
        if not pages:
            continue
        pages_str = ", ".join(pages)
        parts.append(f"{pages_str} ({origin})")
    if parts:
        source_info = "📄출처: " + ", ".join(parts)
    else:
        source_info = "📄출처: 페이지 정보 없음"
else:
    source_info = "📄출처: 페이지 정보 없음"

# -----------------------------
# 5) 최종 출력
# -----------------------------
print(answer)
print(source_info)

자료에 상품권(기프티콘·선불카드 등) 구매·정산에 대한 별도 규정이나 예시는 없습니다.  
따라서 상품권 구매 시 정산 처리 방법은 자료에 없음입니다.
📄출처: 40p, 41p (교육자료), 9조. 정산 시 유의사항 (공통사항) (정산지침)


In [17]:
# 어떤 문서가 검색되었는지 확인
ai_message["source_documents"]

[Document(id='200a19f6-bb8c-4fa1-8992-5d6145539e42', metadata={'images': ['040_img0.png'], 'origin_pdf': '교육자료', 'page_num': 40, 'source': '040.md'}, page_content='11.1.1 현금 또는 예금장부에 입금하고 신용카드장부에서 지출하는 경우의 실제 사례\n===============================================\n\n  \n\n- 무역관 문의: 신용카드 결제내역 일부 중 개인 물품이 실수로 포함되어 결제가 되었습니다. 어떻게 정산  \n처리를 해야할까요?  \n- 무역관 상황: 결제 100불 중 회사분 80불, 개인분 20불 / 카드결제일 : 6월 11일 / 오지출 발견일 : 6월 12일 /  \n신용카드 대금 납부일 : 6월 25일  \n- 처리방법:\n\n  \n\n① (6.11.) 신용카드장부 – 20불 이체(출금) 처리,  \n80불은 기존 실행사업, 비목으로 집행  \n② (6.12.) 무역관 계좌에 20불 입금,  \n송금통화예금장부 - 이체(입금) 처리  \n③ (6.25.) 신용카드장부 – 20불 이체(입금) 처리,  \n송금통화예금장부 – 20불 이체(출금) 처리  \n④ (6.25.) 80불에 대한 카드 대금 처리 : 80불 대체  \n송금통화예금(출금) → 신용카드 장부(입금)\n\n• 현금 또는 예금장부에 입금하고 신용카드장부에서 지출하는 경우의 실제 사례\n=========================================='),
 Document(id='5d545bec-e075-4348-ab82-b88f9b150d5f', metadata={'origin_pdf': '정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)3.  정산 시기 가. 자금정산은 발생주의 원칙에 의거, 집행일에 정산하여

## 5. Retrieval을 위한 keyword 사전 활용
- 직장인이라는 질의가 들어오면, 직장인을 거주자로 자동으로 바꾸도록 설정
- Knowledge Base에서 사용되는 keyword를 활용하여 사용자 질문 수정
- LangChain Expression Language (LCEL)을 활용한 Chain 연계

In [51]:
query = '4직급의 기준을 자세하게 알려주세요. 보기좋게 연번으로 알려주세요.' # 표를 마크다운으로 바꾸면 더 잘 읽는지 확인

In [52]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["기준을 나타내는 표현 -> 경력 기준"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요.
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

In [53]:
new_question = dictionary_chain.invoke({"question": query})

In [54]:
# 바뀐 질의
new_question

'4직급의 경력 기준을 자세하게 알려주세요. 보기 좋게 연번으로 알려주세요.'

In [26]:
ai_response = tax_chain.invoke({"question": query})

In [28]:
print(ai_response['result'])

4직급의 경력 기준은 다음과 같습니다:

1. 행정 분야에서 근무한 6급 공무원으로 5년 이상 경력 소지자
2. 정부투자기관, 경제단체 및 유관기관에서 동일직급 2년 이상 경력 소지자
3. 소령 2년 이상 경력 소지자
4. 민간기업 과장급으로 유관부문 2년 이상 경력 소지자
5. 대학 및 전문학교 전임강사 2년 이상 경력 소지자
6. 전문조사기관의 연구원으로 3년 이상 경력 소지자
7. 해당 분야에 실무경력, 연구 또는 연수 경력자로 당해 직급에 자격이 있다고 인사위원회에서 인정되는 자
8. 기타 전항과 동등한 자격이 있다고 인사위원회에서 인정되는 자
